#Load Web Data Parts

In [37]:
import pandas as pd
import numpy as np
import os

# Load web data parts 
wd1 = pd.read_csv('df_final_web_data_pt_1.txt')
wd2 = pd.read_csv('df_final_web_data_pt_2.txt')

print("WD1 shape:", wd1.shape)
print("WD2 shape:", wd2.shape)
print("\nWD1 columns:", wd1.columns.tolist())
print("WD2 columns:", wd2.columns.tolist())

WD1 shape: (343141, 5)
WD2 shape: (412264, 5)

WD1 columns: ['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time']
WD2 columns: ['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time']


#Quick check

In [21]:
# Preview first rows
print("WD1 head:")
print(wd1.head())

print("\nWD2 head:")
print(wd2.head())

# Check data types and missing values
wd1.info()
wd2.info()

WD1 head:
   client_id            visitor_id                      visit_id process_step  \
0    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   
1    9988021  580560515_7732621733  781255054_21935453173_531117       step_2   
2    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   
3    9988021  580560515_7732621733  781255054_21935453173_531117       step_2   
4    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   

             date_time  
0  2017-04-17 15:27:07  
1  2017-04-17 15:26:51  
2  2017-04-17 15:19:22  
3  2017-04-17 15:19:13  
4  2017-04-17 15:18:04  

WD2 head:
   client_id             visitor_id                      visit_id  \
0     763412  601952081_10457207388  397475557_40440946728_419634   
1    6019349  442094451_91531546617  154620534_35331068705_522317   
2    6019349  442094451_91531546617  154620534_35331068705_522317   
3    6019349  442094451_91531546617  154620534_35331068705_522317 

#Missing value check

In [22]:
import pandas as pd
import numpy as np

wd1 = pd.read_csv('df_final_web_data_pt_1.txt')
wd2 = pd.read_csv('df_final_web_data_pt_2.txt')

wd1['source'] = 'pt1'
wd2['source'] = 'pt2'

web_data = pd.concat([wd1, wd2], ignore_index=True)
print(f"Combined: {web_data.shape}")
print(web_data['source'].value_counts())

Combined: (755405, 6)
source
pt2    412264
pt1    343141
Name: count, dtype: int64


In [23]:
print("Missing values per column:")
missing = web_data.isnull().sum()
print(missing[missing > 0])  # Only problems

print(f"\nRows with ANY missing: {(web_data.isnull().any(axis=1)).sum():,}")
print("Unique steps:", web_data['process_step'].unique())

Missing values per column:
Series([], dtype: int64)

Rows with ANY missing: 0
Unique steps: ['step_3' 'step_2' 'step_1' 'start' 'confirm']


#combining web_data part 1 and 2, adding a source coloumn so the data can be tracked back to orignal source.

In [24]:
# Adding source column BEFORE combining
wd1['source'] = 'pt1'
wd2['source'] = 'pt2'

# Combine
web_data = pd.concat([wd1, wd2], ignore_index=True)
print(f"Combined: {web_data.shape[0]:,} rows")
print("\nSource split:")
print(web_data['source'].value_counts())
print("\nSteps by source:")
print(pd.crosstab(web_data['source'], web_data['process_step']))

Combined: 755,405 rows

Source split:
source
pt2    412264
pt1    343141
Name: count, dtype: int64

Steps by source:
process_step  confirm   start  step_1  step_2  step_3
source                                               
pt1             45403  108910   73432   61768   53628
pt2             57560  135035   89761   71294   58614


#Quick Check to see the merge 

In [25]:
print("=== CURRENT WEB DATA STATUS ===")
print(f"Shape: {web_data.shape}")
print(f"Columns: {web_data.columns.tolist()}")
print("\nSample (first 5 rows):")
print(web_data.head())
print("\nData types:")
print(web_data.dtypes)
print("\nDate range:", web_data['date_time'].min(), "→", web_data['date_time'].max())

=== CURRENT WEB DATA STATUS ===
Shape: (755405, 6)
Columns: ['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time', 'source']

Sample (first 5 rows):
   client_id            visitor_id                      visit_id process_step  \
0    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   
1    9988021  580560515_7732621733  781255054_21935453173_531117       step_2   
2    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   
3    9988021  580560515_7732621733  781255054_21935453173_531117       step_2   
4    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   

             date_time source  
0  2017-04-17 15:27:07    pt1  
1  2017-04-17 15:26:51    pt1  
2  2017-04-17 15:19:22    pt1  
3  2017-04-17 15:19:13    pt1  
4  2017-04-17 15:18:04    pt1  

Data types:
client_id        int64
visitor_id      object
visit_id        object
process_step    object
date_time       object
source          object
dtype: 

#Cleaning

In [26]:
# Fix datetime 
web_data['date_time'] = pd.to_datetime(web_data['date_time'], format='%Y-%m-%d %H:%M:%S')
print("Datetime fixed:", web_data['date_time'].dtype)
print("Range:", web_data['date_time'].min().strftime('%Y-%m-%d'), '→', 
      web_data['date_time'].max().strftime('%Y-%m-%d'))

Datetime fixed: datetime64[ns]
Range: 2017-03-15 → 2017-06-20


In [27]:
print("Duplicates check...")
dups = web_data.duplicated(subset=['visit_id', 'process_step', 'date_time']).sum()
web_data_clean = web_data.drop_duplicates(subset=['visit_id', 'process_step', 'date_time']).reset_index(drop=True)
print(f"Removed {dups:,} duplicates → {len(web_data_clean):,} rows")

Duplicates check...
Removed 10,764 duplicates → 744,641 rows


In [28]:
# Check if duplicates cross pt1/pt2 
cross_source_dups = web_data.groupby(['visit_id', 'process_step', 'date_time'])['source'].nunique()
print("Cross-source duplicates:", (cross_source_dups > 1).sum())
print("All duplicates are WITHIN same source ")

Cross-source duplicates: 0
All duplicates are WITHIN same source 


In [29]:
# Sample 3 duplicates + their source, check to see if the dupicats are from the same source or not.
dup_samples = web_data[web_data.duplicated(subset=['visit_id', 'process_step', 'date_time'], keep=False)].head(6)
print("Duplicate samples (same source):")
print(dup_samples[['visit_id', 'process_step', 'date_time', 'source']])

Duplicate samples (same source):
                         visit_id process_step           date_time source
364  223297395_36250329195_832161        start 2017-04-28 12:27:28    pt1
365  223297395_36250329195_832161        start 2017-04-28 12:27:28    pt1
367  688984457_43441834354_912755        start 2017-04-28 14:20:01    pt1
368  688984457_43441834354_912755        start 2017-04-28 14:20:01    pt1
382  330543236_29863358529_771432        start 2017-04-28 02:52:29    pt1
383  330543236_29863358529_771432        start 2017-04-28 02:52:29    pt1


In [30]:
# Remove duplicates (keeps first occurrence)
web_data = web_data.drop_duplicates(subset=['visit_id', 'process_step', 'date_time'])
print(f"Cleaned: {len(web_data)} rows")

Cleaned: 744641 rows


In [31]:
# sort by visit_id and date_time to ensure chronological order
web_data_clean = web_data_clean.sort_values(['visit_id', 'date_time']).reset_index(drop=True)
print("Chronological order")
print("\nFirst visit sample:")
print(web_data_clean[web_data_clean['visit_id'] == web_data_clean['visit_id'].iloc[0]])

Chronological order

First visit sample:
   client_id            visitor_id                      visit_id process_step  \
0    3561384  451664975_1722933822  100012776_37918976071_457913      confirm   
1    3561384  451664975_1722933822  100012776_37918976071_457913      confirm   

            date_time source  
0 2017-04-26 13:22:17    pt1  
1 2017-04-26 13:23:09    pt1  


In [ ]:
# steps for data
expected_steps = {'start', 'step_1', 'step_2', 'step_3', 'confirm'}
full_funnels = web_data_clean.groupby('visit_id')['process_step'].apply(
    lambda x: set(x.unique()) == expected_steps
).sum()
print(f"True full funnels: {full_funnels} / {web_data_clean['visit_id'].nunique()}")

True full funnels: 76998 / 158095


In [40]:

#  STRING CLEAN (ONLY) 
web_data_clean['process_step'] = web_data_clean['process_step'].str.strip().str.lower()

# VERIFY CATEGORIES
print("9. Clean steps:", sorted(web_data_clean['process_step'].unique()))
print("Sources:", sorted(web_data_clean['source'].unique()))

# EXPORT
os.makedirs('data_cleaned', exist_ok=True)
web_data_clean.to_pickle('data_cleaned/web_data_cleaned.pkl')
web_data_clean.to_csv('data_cleaned/web_data_cleaned.csv', index=False)
print("\nCLEANING 100% COMPLETE!")
print("Final shape:", web_data_clean.shape)

9. Clean steps: ['confirm', 'start', 'step_1', 'step_2', 'step_3']
Sources: ['pt1', 'pt2']

CLEANING 100% COMPLETE!
Final shape: (744641, 6)


In [ ]:

print("Before sort - first 3:")
print(web_data_clean.head(3))

web_data_clean.sort_values('date_time', inplace=True)
print("\nAfter sort - first 3:")
print(web_data_clean.head(3))

# Save sorted version
web_data_clean.to_csv('data_cleaned/web_funnel_sorted.csv', index=False)
print("\nSorted CSV created!")

Before sort - first 3:
   client_id             visitor_id                      visit_id  \
0    3561384   451664975_1722933822  100012776_37918976071_457913   
1    3561384   451664975_1722933822  100012776_37918976071_457913   
2    9056452  306992881_89423906595     1000165_4190026492_760066   

  process_step           date_time source  
0      confirm 2017-04-26 13:22:17    pt1  
1      confirm 2017-04-26 13:23:09    pt1  
2        start 2017-06-04 01:07:29    pt2  

After sort - first 3:
        client_id             visitor_id                      visit_id  \
265216    9088444  242404224_96732670250  423038079_46067236368_400417   
134323    7179755  167765295_97487764427   264484508_5982901710_928530   
134324    7179755  167765295_97487764427   264484508_5982901710_928530   

       process_step           date_time source  
265216       step_3 2017-03-15 00:03:03    pt1  
134323        start 2017-03-15 00:19:28    pt1  
134324       step_1 2017-03-15 00:20:50    pt1  

Sorted 

In [42]:
# CURRENT STATE: already sorted in memory
web_data_clean.to_pickle('data_cleaned/web_funnel_sorted.pkl')
print("Sorted pickle saved!")
print("4 files complete:")
print("- web_funnel_final.csv (raw)")
print("- web_funnel_sorted.csv (chrono CSV)")
print("- web_funnel_final.pkl (raw fast)")
print("- web_funnel_sorted.pkl (chrono fast) ✓")

Sorted pickle saved!
4 files complete:
- web_funnel_final.csv (raw)
- web_funnel_sorted.csv (chrono CSV)
- web_funnel_final.pkl (raw fast)
- web_funnel_sorted.pkl (chrono fast) ✓
